# 01 — Dataset Audit

Coverage, gaps, and the thinnest nutrients across `data/food/bd_food.csv`,
`data/recipe/`, and `data/lifestyle/exercises.csv`.

> Run `scripts/validate_datasets.py` first — this notebook assumes structurally valid data.
> While the CSVs contain DEMO rows, the numbers here describe placeholder data only.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.diet.food_db import NUTRIENT_COLUMNS, load_foods
from src.diet.recipe import load_recipes
from src.diet.food_db import food_index
from src.lifestyle.exercise_db import load_exercises

foods = load_foods()
recipes = load_recipes(food_index(foods))
exercises = load_exercises()
len(foods), len(recipes), len(exercises)

## Food-group coverage
How many foods per group — where the dataset is thin.

In [ ]:
from collections import Counter

group_counts = Counter(f.food_group for f in foods)
for group, count in group_counts.most_common():
    print(f"{group:12s} {count}")

## Thinnest nutrients
Share of foods providing a meaningful amount (> 5% of the female-adult RDI per 100 g)
of each tracked nutrient — low percentages mark nutrients the food list barely covers.

In [ ]:
from src.core.config import REFERENCE_DIR, load_yaml

rdi = load_yaml(REFERENCE_DIR / "rdi_values.yaml")["groups"]["female_adult"]
coverage = {}
for col in NUTRIENT_COLUMNS:
    target = rdi.get(col)
    if not target:
        continue
    providers = sum(1 for f in foods if f.nutrients[col] > 0.05 * target)
    coverage[col] = providers / len(foods) if foods else 0
for col, share in sorted(coverage.items(), key=lambda kv: kv[1]):
    print(f"{col:14s} {share:5.0%}")

## Exercise catalog balance
Category × intensity spread — the FITT-VP builder needs choices in every cell it draws from.

In [ ]:
matrix = Counter((e.category, e.intensity_level) for e in exercises)
for (category, level), count in sorted(matrix.items()):
    print(f"{category:12s} {level:9s} {count}")